IDEAS

1. Find out what the graphical interpretation of the Hamming weight could be here (pretty sure it should be simple)
2. Generalise this to any kind of degree

In [ ]:
import igraph as ig
import torch as tc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

from tqdm import tqdm

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

from src.automata import LLNA
from src.simulation import *

from src.analysis import hamming_weight, boolean_sens, binary_indices

# Mean-Field Propagation of densities in LLNAs

Generally, we need to run the simulation in order to find out how the states in an LLNA develop. However, by simply looking at the average densities in the network, the degree distribution, and the local update rule, we can estimate how the densities will evolve on average. The same approach occurs in cellular automata.

Note that this mean-field approach is particularly useful in the context of LLNAs because its local update rules are totalistic.

First consider a simple case: a ring network with a random initial configuration and a simple local update rule

In [ ]:
# verify for some obvious cases: minimum

resolution = 3
fig, axs = plt.subplots(4,1,figsize=(10,6))

# assign rules and generate LLNA
born_if = []
survive_if = [0, 1, 2]
model0 = LLNA(resolution, x=born_if, y=survive_if, iso=True)
model0.diagram(ax=axs[0])
axs[0].set_title(fr"Static life")

# assign rules and generate LLNA
born_if = [0, 1, 2]
survive_if = []
model1 = LLNA(resolution, x=born_if, y=survive_if, iso=True)
model1.diagram(ax=axs[1])
axs[1].set_title(fr"Periodic life")

# assign rules and generate LLNA
born_if = [1]
survive_if = [1]
model2 = LLNA(resolution, x=born_if, y=survive_if, iso=True)
model2.diagram(ax=axs[2])
axs[2].set_title(fr"Converge to average density?")

# assign rules and generate LLNA
born_if = [2]
survive_if = [1,2]
model3 = LLNA(resolution, x=born_if, y=survive_if, iso=True)
model3.diagram(ax=axs[3])
axs[3].set_title(fr"But what here, though?")

fig.suptitle(fr"Some predictable and less predictable LLNAs")
fig.tight_layout()

Let us see how this actually works out in a density time series on a ring network.

In [ ]:
def init_config_with_dens(N, dens):
    # defines a random initial configuration with a fixed state density
    s0 = np.zeros(N, dtype=float)
    s0[:np.round(dens*N).astype(int)] = 1.
    np.random.shuffle(s0)
    return s0

In [ ]:
# show some density paths for a couple of initial conditions for a particular LLNA

N = 100
G = ig.Graph.Ring(N)
T = 100

# find edges
G.to_directed()
edges = tc.tensor(G.get_edgelist()).T
G.to_undirected()

s_list=[]
init_rhos = np.arange(0.01, 1., 0.001)
# random init config
for init_rho in tqdm(init_rhos):
    s0 = init_config_with_dens(N, init_rho)
    s0 = tc.tensor(s0)[None,:]

    # evolve first model
    s = model3.forward(edges, s0, T=T)
    s = np.array(s)[0]
    s_mean = np.mean(s, axis=1)
    s_list.append(s)
    plt.plot(s_mean, label=fr'$\rho={round(init_rho,1)}$', alpha=.005, color='maroon')
# plt.legend(ncol=3)
plt.ylabel("Density")
plt.xlabel("Time step")
plt.title("Density evolutions from different initial densities")

s_array = np.array(s_list)

In [ ]:
# np.mean(s_array,axis=2)

s_array.shape

plt.plot(list(init_rhos), np.mean(s_array, axis=2)[:,-1])

In [ ]:
plt.imshow(s_array[np.random.randint(0, 900)])

IDEA: with these kinds of , I would assume that ...
1. complex networks solidify a little less quickly than cellular automata
2. 

In [ ]:
rho_t_array = np.mean(s_array[:,0], axis=1)
rho_tplus_array = np.mean(s_array[:,1], axis=1)

fontsize=18

plt.scatter(rho_t_array, rho_tplus_array, alpha=0.1)

plt.title("Next-step density from random initial configuration", fontsize=fontsize)
plt.xlabel(r"$\bar{\rho}^t$", fontsize=fontsize)
plt.ylabel(r"$\bar{\rho}^{t+1}$", fontsize=fontsize)

plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])

We can understand this behaviour in a course-grained fashion by looking at the average densities. This clearly (and importantly) overlooks the fact that generally local structures will form quite fast, such that coarse-graining has little predictive power. But let's see where it brings us, nonetheless.

- In an initial random configuration of exactly $N/2$ "alive" nodes, there is a probability of $50$ percent of encountering an alive node.
- If the network is sufficiently large, then approximately $50$ percent of the neighbouring cells will be alive
- More specifically, for a ring network, there is a $1/4$ probability of all of the neighbours being alive (dead), and a $1/2$ probability of half of the neighbours being alive (dead).
- If you're dead, a particular response is triggered. If the node is alive, another response is triggered.

For a random node with index $i$ with $2$ neighbours (in the ring network), therefore, the expectation value of its state evolves in a mean-field fashion as follows:
$$
\mathbb{E}(s_i^{t+1}) = \bar{\rho}^t \left( \dfrac{1}{4} S_0 + \dfrac{2}{4} S_1 + \dfrac{1}{4} S_2 \right) + (1-\bar{\rho}^t)\left( \dfrac{1}{4} B_0 + \dfrac{2}{4} B_1 + \dfrac{1}{4} B_2 \right),
$$
where $B_i$ (resp. $S_i$) is $1$ if a cell is born (reps. survives) if it has a density in interval $i$, and $0$ otherwise. The average density of the system at time $t$, $\bar{\rho}^t$ is taken to be equal to the average state of any given node, so $\mathbb{E}(s_i^t) = \bar{\rho}^t = \sum_{i=1}^N \rho_i^t$.

So, for $S_0 = S_2 = B_0 = B_2 = 0$ and $S_1 = B_1 = 1$, we find
$$
\bar{\rho}^{t+1} = \bar{\rho}^t \left(\dfrac{2}{4}\right) + (1-\bar{\rho}^t)\left(\dfrac{2}{4}\right) = 1/2,
$$
such that there is a 50/50 percent chance of finding a living node in the network.

This is in general not trivial, however, because the probabilities of encountering particular densities also changes over time:
$$
\bar{\rho}^{t+1} = \bar{\rho}^t \sum_{j=0}^{R-1}P(\rho^t \in R_j)S_j + (1-\bar{\rho}^t)\sum_{j=0}^{R-1}P(\rho^t \in R_j)B_j,
$$
or, more compactly
\begin{align}
\boxed{\bar{\rho}^{t+1} =  \sum_{j=0}^{R-1}P(\rho^t \in R_j)\left\{ \bar{\rho}^t S_j + (1-\bar{\rho}^t) B_j\right\}}
\end{align}
The factor $P(\rho^t \in R_j)$ represents the probability of encountering at time $t$ a local neighbourhood density $\rho^t$ that is in the density interval $R_j$, given an average network density of $\bar{\rho}^t$. Here the terms $B_j$ and $S_j$ are fixed by the local update rule, but the probabilities $P$ generally change over time. In particular, $P$ is a binomial distribution, with
$$
P(\rho^t \in R_j) = \binom{R-1}{j} (\bar{\rho}^t)^j (1-\bar{\rho}^t)^{R-j-1},
$$
where $R-1 = d=2$ is the degree (which is fixed to $2$ for ring networks), and $j \in \{0, 1, 2\}$. Note that for general networks, this probability depends on the (varying) node degree. Below some of these probabilities are shown.

Below we show some randomly chosen LLNA

In [ ]:
# code this recurrent behaviour
from scipy.stats import binom

def mean_field_dens_propagation(dens, resolution, B_set, S_set):
    # NOTE: this is only for the case where d = resolution - 1
    B_set_binary = np.zeros(resolution)
    S_set_binary = np.zeros(resolution)
    B_set_binary[B_set] = 1
    S_set_binary[S_set] = 1
    new_dens = 0
    for j in range(resolution):
        prob = binom.pmf(j, resolution-1, dens)
        update = dens * S_set_binary[j] + (1-dens) * B_set_binary[j]
        # make sure the density is between 0 and 1
        product = prob * update
        new_dens += product
    return np.clip(new_dens, 0, 1)

def cobweb_list(init_dens, update_fun, T):
    # create two lists that can used to plot a cobweb plot
    # inputs are: initial density, the update function with a single numeric argument, and the number of time steps
    # Start list on the update curve
    dens_x_new = init_dens
    dens_y_new = update_fun(init_dens)
    dens_x_list = [dens_x_new]
    dens_y_list = [dens_y_new]
    for _ in range(T-1):
        # move to first bisector
        dens_x_list.append(dens_y_new)
        dens_y_list.append(dens_y_new)
        # update value according to update function
        dens_x_new = dens_y_new
        dens_y_new = update_fun(dens_y_new)
        # move to new point on update curve
        dens_x_list.append(dens_x_new)
        dens_y_list.append(dens_y_new)
    return dens_x_list, dens_y_list

In [ ]:
dens_list = np.linspace(0,1,101)
resolution = 3

import itertools
R_set = np.arange(resolution)
all_possible_subsets = list(itertools.chain.from_iterable(itertools.combinations(R_set, r) for r in range(resolution+1)))

B_set = list(all_possible_subsets[np.random.randint(2**resolution)])
S_set = list(all_possible_subsets[np.random.randint(2**resolution)])
new_dens_list = [mean_field_dens_propagation(dens, resolution, B_set, S_set) for dens in dens_list]

fig, axs = plt.subplots(1,2,figsize=(10,5))
fontsize=18

axs[0].plot(dens_list, new_dens_list, lw=3)
axs[0].set_title(f"Analytical with cobweb plot", fontsize=fontsize)

axs[0].set_xlabel(r"$\bar{\rho}^t$", fontsize=fontsize)
axs[0].set_ylabel(r"$\bar{\rho}^{t+1}$", fontsize=fontsize)

from functools import partial
update_fun = partial(mean_field_dens_propagation, resolution=resolution, B_set=B_set, S_set=S_set)

init_dens = .5
T = 6
dens_x_list, dens_y_list = cobweb_list(init_dens, update_fun, T)

ddens = .2
for dens_init in np.arange(0, 1+ddens, ddens):
    dens_x_list, dens_y_list = cobweb_list(dens_init, update_fun, T)
    axs[0].plot(dens_x_list, dens_y_list, ls='--', label=f"Init dens. {round(dens_init,1)}")
axs[0].plot([0, 1], [0, 1], ls=':', color='tab:blue', alpha=.5)
axs[0].legend()
# plt.grid(True)

# let's compare this to an empirical distribution
# show some density paths for a couple of initial conditions for a particular LLNA

N = 1001
G = ig.Graph.Ring(N)
T = 1

# find edges
G.to_directed()
edges = tc.tensor(G.get_edgelist()).T
G.to_undirected()

# get model with same params as above
model = LLNA(resolution, x=B_set, y=S_set, iso=True)

s_list=[]
init_rhos = np.arange(0, 1.+0.001, 0.001)
# random init config
for init_rho in tqdm(init_rhos):
    s0 = init_config_with_dens(N, init_rho)
    s0 = tc.tensor(s0)[None,:]
    # evolve model TODO this can be much more efficient!
    s = model.forward(edges, s0, T=T)
    s = np.array(s)[0]
    s_list.append(s)
s_array = np.array(s_list)

rho_t_array = np.mean(s_array[:,0], axis=1)
rho_tplus_array = np.mean(s_array[:,1], axis=1)

axs[1].scatter(rho_t_array, rho_tplus_array, alpha=0.1, label='From simulations')
axs[1].plot(dens_list, new_dens_list, color='k', ls='--', label='From calculation')
axs[1].plot([0, 1], [0, 1], ls=':', color='tab:blue', alpha=.5)

legend=plt.legend(fontsize=fontsize)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[1].set_title(f"On ring network with {N} nodes", fontsize=fontsize)
axs[1].set_xlabel(r"$\bar{\rho}^t$", fontsize=fontsize)
axs[1].set_ylabel(r"$\bar{\rho}^{t+1}$", fontsize=fontsize)

axs[1].set_xlim([-0.05, 1.05])
axs[1].set_ylim([-0.05, 1.05])

axs[0].set_aspect(1)
axs[1].set_aspect(1)

fig.suptitle(f"Next-step density from random initial configuration for rule {model.__str__(latex=True)}", fontsize=fontsize+2)

fig.tight_layout()

This is nice but not really surprising; I think the only stochasticity we see here is due to the binomial distribution over a small neighbourhood.
- I could make plots that look a little further into the future (t+2 or t+3)
- I could check the influence of number of nodes and neighbourhood size (which is going to be substantial)

In [ ]:
# Let us look one step beyond
dens_list = np.linspace(0,1,101)
resolution = 3

import itertools
R_set = np.arange(resolution)
all_possible_subsets = list(itertools.chain.from_iterable(itertools.combinations(R_set, r) for r in range(resolution+1)))

B_set = [2] # list(all_possible_subsets[np.random.randint(2**resolution)])
S_set = [1,2] # list(all_possible_subsets[np.random.randint(2**resolution)])

# repeat for two time steps
new_dens_list = [mean_field_dens_propagation(dens, resolution, B_set, S_set) for dens in dens_list]
new_new_dens_list = [mean_field_dens_propagation(new_dens, resolution, B_set, S_set) for new_dens in new_dens_list]

fig, axs = plt.subplots(1,2,figsize=(10,5))
fontsize=18

axs[0].plot(dens_list, new_new_dens_list, lw=3)

axs[0].set_xlabel(r"$\bar{\rho}^t$", fontsize=fontsize)
axs[0].set_ylabel(r"$\bar{\rho}^{t+2}$", fontsize=fontsize)

# let's compare this to an empirical distribution
# show some density paths for a couple of initial conditions for a particular LLNA

N = 10001
G = ig.Graph.Ring(N)
T = 2

# find edges
G.to_directed()
edges = tc.tensor(G.get_edgelist()).T
G.to_undirected()

# get model with same params as above
model = LLNA(resolution, x=B_set, y=S_set, iso=True)

s_list=[]
init_rhos = np.arange(0, 1.+0.001, 0.001)
# random init config
for init_rho in tqdm(init_rhos):
    s0 = init_config_with_dens(N, init_rho)
    s0 = tc.tensor(s0)[None,:]
    # evolve model TODO this can be much more efficient!
    s = model.forward(edges, s0, T=T)
    s = np.array(s)[0]
    s_list.append(s)
s_array = np.array(s_list)

rho_t_array = np.mean(s_array[:,0], axis=1)
rho_tplusplus_array = np.mean(s_array[:,2], axis=1)

axs[1].scatter(rho_t_array, rho_tplusplus_array, alpha=0.1, label='From simulations')
axs[1].plot(dens_list, new_new_dens_list, color='k', ls='--', label='From calculation')
axs[1].plot([0, 1], [0, 1], ls=':', color='tab:blue', alpha=.5)

legend=plt.legend(fontsize=fontsize)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[1].set_title(f"On ring network with {N} nodes", fontsize=fontsize)
axs[1].set_xlabel(r"$\bar{\rho}^t$", fontsize=fontsize)
axs[1].set_ylabel(r"$\bar{\rho}^{t+2}$", fontsize=fontsize)

axs[1].set_xlim([-0.05, 1.05])
axs[1].set_ylim([-0.05, 1.05])

axs[0].set_aspect(1)
axs[1].set_aspect(1)

fig.suptitle(f"Next-next-step density from random initial configuration for rule {model.__str__(latex=True)}", fontsize=fontsize+2)

fig.tight_layout()

**Fixed points**

The analytical expression above allows us to identify fixed points of this system, as well as their stability. Note the following:
1. If the average density is fixed, we find that $\bar{\rho}^{t+1} = \bar{\rho}^t$. Let's call this $\bar{\rho}^*$.
2. If this fixed point is stable, small perturbations should cancel out. If it is unstable, small perturbations will amplify. This translates to the following demands:
    - $\left|\dfrac{d\bar{\rho}^{t+1}}{d\bar{\rho}^{t}}\right|_{\bar{\rho}^t = \bar{\rho}^*} < 1$: stable fixed point
    - $\left|\dfrac{d\bar{\rho}^{t+1}}{d\bar{\rho}^{t}}\right|_{\bar{\rho}^t = \bar{\rho}^*} > 1$: unstable fixed point
    - $\left|\dfrac{d\bar{\rho}^{t+1}}{d\bar{\rho}^{t}}\right|_{\bar{\rho}^t = \bar{\rho}^*} = 1$: borderline stable fixed point
3. It is also possible to find closed loops (periodic mean-field behaviour), by demanding that $\bar{\rho}^{t+2} = \bar{\rho}^t \neq \bar{\rho}^{t+1}$ and using the recursive qualities of the expression.
4. This can be generalised to loops with a longer 'wavelength', but I presume the math becomes quite involved quite quickly.

Let us first find the general expression for the fixed point $\rho^*$.

\begin{align*}
\bar{\rho}^* =  \sum_{j=0}^{R-1}P(\rho \in R_j)\left\{ \bar{\rho}^* S_j + (1-\bar{\rho}^*) B_j\right\}, \qquad \qquad P(\rho \in R_j) = \binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1},
\end{align*}

so we want to solve
$$
\sum_{j=0}^{R-1}\binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1}\left\{ \bar{\rho}^* S_j + (1-\bar{\rho}^*) B_j\right\} = \bar{\rho}^*
$$
which may be re-written as
$$
\bar{\rho}^* \mathcal{S}(\bar{\rho}^*) + (1-\bar{\rho}^*)\mathcal{B}(\bar{\rho}^*) = \bar{\rho}^*
$$
with the truncated binomial sums
\begin{align}
\begin{cases}
    \mathcal{S}(\bar{\rho}^*) = \sum_{j=0}^{R-1}\binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1} S_j = \sum_{j:S_j=1}^{R-1}\binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1} \in [0, 1], \\
    \mathcal{B}(\bar{\rho}^*) = \sum_{j=0}^{R-1}\binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1} B_j = \sum_{j:B_j=1}^{R-1}\binom{R-1}{j} (\bar{\rho}^*)^j (1-\bar{\rho}^*)^{R-j-1} \in [0, 1].
\end{cases}
\end{align}
Depending on the values of the $S_j$ and $B_j$, this can become a difficult polynomial to solve for its real roots between $0$ and $1$. There are some special cases: if all density intervals are activated, the binomial sum is complete, such that $\mathcal{S}(\bar{\rho}^*) = 1$. If none are activated, the sum is zero. So:
- All survive, none are born: $\bar{\rho}^* = \bar{\rho}^*$: any initial density is a fixed point
- None survive, all are born: $\bar{\rho}^* = 0.5$: a density of 1/2 is a fixed point
- All survive, all are born: $\bar{\rho}^* = 1$: all nodes will become alive
- None surive, none are born: $\bar{\rho}^* = 0$: all nodes will die

We will find the solutions in a numerical way using `sympy`.

In [ ]:
from sympy import symbols, binomial, solve, simplify, diff, nsolve
from sympy import Eq, Interval

def find_fixed_points(resolution, B_set, S_set, eval=False, return_stability=False):
    # TODO: I'm not sure whether this works well for resolution != 3
    # Define the symbols
    rho = symbols('rho')

    # Explicitly compute the sums for S and B
    S_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in S_set)
    B_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in B_set)

    # find derivative of mean-field equation
    rhs = rho*S_sum + (1-rho)*B_sum
    rhs_dot = diff(rhs, rho)

    # fixed-point mean-field equation
    # NOTE: this is quite a nice expression, mostly
    fixed_point_eq = simplify(Eq(rho, rhs))

    # check special case of equation that always holds
    # in that case all densities are fixed points, but we output only one.
    if (fixed_point_eq == True):
        if return_stability: return [.5], [True]
        else: return [.5]

    # find valid roots
    all_roots = solve(fixed_point_eq, rho)
    all_roots = [simplify(root) for root in all_roots]
    valid_roots = [root for root in all_roots if root.is_real and Interval(0, 1).contains(root)]
    # add numerical solutions in case none were found analytically
    if not len(valid_roots):
        all_roots = {nsolve(fixed_point_eq, rho, start_rho) for start_rho in [0, .5, 1]}
        valid_roots = [root for root in all_roots if root.is_real and Interval(0, 1).contains(root)]
    # check whether the fixed point is stable
    stable_roots = [np.abs(rhs_dot.subs(rho, root))<1 for root in valid_roots]
    if return_stability:
        if eval:
            return [valid_root.evalf() for valid_root in valid_roots], stable_roots
        return valid_roots, stable_roots
    else:
        if eval:
            return [valid_root.evalf() for valid_root in valid_roots]
        return valid_roots

B_set = [0, 1]
S_set = []

rho = symbols('rho')

# Explicitly compute the sums for S and B
S_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in S_set)
B_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in B_set)

# find derivative of mean-field equation
rhs = rho*S_sum + (1-rho)*B_sum
rhs_dot = diff(rhs, rho)

# fixed-point mean-field equation
fixed_point_eq = simplify(Eq(rho, rhs))

In [ ]:
# S_set = all_possible_subsets[np.random.randint(2**resolution)]
# B_set = all_possible_subsets[np.random.randint(2**resolution)]

# Explicitly compute the sums for S and B
# S_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in S_set)
# B_sum = sum(binomial(resolution - 1, j) * rho**j * (1 - rho)**(resolution - 1 - j) for j in B_set)

# fixed-point mean-field equation
# fixed_point_eq = simplify(Eq(rho*S_sum + (1-rho)*B_sum, rho))
# fixed_point_eq

# fixed_point_eq_ = Eq(rho*S_sum + (1-rho)*B_sum, 0)


resolution=5
B_set = [0, 1, 4]
S_set = [0, 1, 3, 4]
fixed_points, stability = find_fixed_points(resolution, B_set, S_set, eval=True, return_stability=True)

fixed_points, stability

In [ ]:
fig, axs = plt.subplots(8,8,figsize=(15,15), sharex=True, sharey=True)

dens_list = np.linspace(0,1,101)
resolution = 3

all_possible_subsets = list(itertools.chain.from_iterable(itertools.combinations(R_set, r) for r in range(resolution+1)))
for axh, B_set in tqdm(zip(axs, all_possible_subsets), total=2**resolution):
    for ax, S_set in zip(axh, all_possible_subsets):
        new_dens_list = [mean_field_dens_propagation(dens, resolution, list(B_set), list(S_set)) for dens in dens_list]

        ax.plot(dens_list, new_dens_list, lw=3, zorder=0)
        ax.set_title(f"B: {list(B_set)}, S: {list(S_set)}", size=10)
        ax.set_xlim([-0.05,1.05])
        ax.set_ylim([-0.05,1.05])

        # plot some paths
        update_fun = partial(mean_field_dens_propagation, resolution=resolution, B_set=list(B_set), S_set=list(S_set))
        T = 5
        ddens = 0.2
        for dens_init in np.arange(0, 1+ddens, ddens):
            dens_x_list, dens_y_list = cobweb_list(dens_init, update_fun, T)
            ax.plot(dens_x_list, dens_y_list, ls='--', color='tab:blue', alpha=.5, zorder=1)

        # find and plot fixed points
        fixed_points, stability = find_fixed_points(resolution, B_set, S_set, eval=True, return_stability=True)
        for fixed_point, stab in zip(fixed_points, stability):
            if stab: color='limegreen'
            else: color='red'
            ax.scatter(fixed_point, fixed_point, c=color, marker='o', zorder=2)


fig.suptitle("Mean-field behaviour of average densities in a ring network for all resolution-3 LLNAs", size=20)
fig.supxlabel(fr"Current average density $\bar{{\rho}}^t$")
fig.supylabel(fr"New average density $\bar{{\rho}}^{{t+1}}$")
fig.tight_layout()

In [ ]:
model = LLNA(3, x=[2], y=[1,2], iso=True)
model.diagram()

In [ ]:
# compare two random isomorphic LLNAs
resolution = 3
B_set = [2] # all_possible_subsets[np.random.randint(2**resolution)]
S_set = [1,2] # all_possible_subsets[np.random.randint(2**resolution)]

def comp_of_mirror(resolution, intervals):
    # take mirror intervals
    mirror = resolution - 1 - np.array(intervals)
    # take complement set
    comp_mirror = list(set(range(resolution))-set(mirror))
    return comp_mirror

B_set_iso = comp_of_mirror(resolution, S_set)
S_set_iso = comp_of_mirror(resolution, B_set)

fig, axs = plt.subplots(1,2,figsize=(10,4))

new_dens_list = [mean_field_dens_propagation(dens, resolution, list(B_set), list(S_set)) for dens in dens_list]
new_dens_list_iso = [mean_field_dens_propagation(dens, resolution, list(B_set_iso), list(S_set_iso)) for dens in dens_list]

axs[0].plot(dens_list, new_dens_list, lw=3, zorder=0)
axs[0].set_title(f"B: {list(B_set)}, S: {list(S_set)}", size=10)

axs[1].plot(dens_list, new_dens_list_iso, lw=3, zorder=0)
axs[1].set_title(f"B: {list(B_set_iso)}, S: {list(S_set_iso)}", size=10)

for ax in axs:
    ax.set_xlim([-0.05,1.05])
    ax.set_ylim([-0.05,1.05])

# plot some paths
update_fun = partial(mean_field_dens_propagation, resolution=resolution, B_set=list(B_set), S_set=list(S_set))
update_fun_iso = partial(mean_field_dens_propagation, resolution=resolution, B_set=list(B_set_iso), S_set=list(S_set_iso))

T = 5
ddens = 0.2
for dens_init in np.arange(0, 1+ddens, ddens):
    dens_x_list, dens_y_list = cobweb_list(dens_init, update_fun, T)
    axs[0].plot(dens_x_list, dens_y_list, ls='--', color='tab:blue', alpha=.5, zorder=1)
    dens_x_list_iso, dens_y_list_iso = cobweb_list(dens_init, update_fun_iso, T)
    axs[1].plot(dens_x_list_iso, dens_y_list_iso, ls='--', color='tab:blue', alpha=.5, zorder=1)

# find and plot fixed points
fixed_points, stability = find_fixed_points(resolution, B_set, S_set, eval=True, return_stability=True)
fixed_points_iso, stability_iso = find_fixed_points(resolution, B_set_iso, S_set_iso, eval=True, return_stability=True)
for fixed_point, stab, fixed_point_iso, stab_iso in zip(fixed_points, stability, fixed_points_iso, stability_iso):
    if stab: color='limegreen'
    else: color='red'
    axs[0].scatter(fixed_point, fixed_point, c=color, marker='o', zorder=2)
    if stab_iso: color_iso='limegreen'
    else: color_iso='red'
    axs[1].scatter(fixed_point_iso, fixed_point_iso, c=color_iso, marker='o', zorder=2)

To do next:
1. inspect some of these plots with actual paths from cellular automata
2. analytically find the stable points of these systems
3. find the isomorphic cases and compare their density curves

In [ ]:
# code this recurrent behaviour
resolution = 3
T = 150

import itertools
R_set = np.arange(resolution)
all_possible_subsets = list(itertools.chain.from_iterable(itertools.combinations(R_set, r) for r in range(resolution+1)))

# determine initial density that is on a fixed point
init_dens = 0
init_stab = True
while (init_dens ==0 or init_dens ==1):
    B_set = [2] # list(all_possible_subsets[np.random.randint(2**resolution)])
    S_set = [1,2] # list(all_possible_subsets[np.random.randint(2**resolution)])
    fixed_points, stability = find_fixed_points(resolution, B_set, S_set, eval=True, return_stability=True)
    for fixed_point, stab in zip(fixed_points, stability):
        if fixed_point > 0 and fixed_point < 1:
            init_dens = float(fixed_point)
            init_stab = stab

update_fun = partial(mean_field_dens_propagation, resolution=resolution, B_set=B_set, S_set=S_set)
densities_over_time = cobweb_list(init_dens, update_fun, T)[0][::2]

fig, axs = plt.subplots(2,1,figsize=(12,5))

if init_stab:
    stab_str = 'stable'
else:
    stab_str = 'unstable'
axs[1].set_title(f"B_set: {list(B_set)}, S_set: {list(S_set)}, initial fixed-point density: {round(init_dens,2)} ({stab_str})")
axs[1].plot(densities_over_time, lw=3, label="Mean-field approach")
axs[1].set_ylim([-0.05,1.05])

# plot actual densities from simulations
N = 100
G = ig.Graph.Ring(N)
# find edges
G.to_directed()
edges = tc.tensor(G.get_edgelist()).T
G.to_undirected()

# make model
model = LLNA(resolution, x=B_set, y=S_set, iso=True)

# make realisations from various initial conditions with identical initial density
iters = 100
rhos = []
for _ in range(iters):
    s0 = init_config_with_dens(N, init_dens)
    s0 = tc.tensor(s0)[None,:]

    # evolve first model
    s = model.forward(edges, s0, T=T)
    s = np.array(s)[0]
    rho = np.mean(s, axis=1)
    rhos.append(rho)

# plot on same ax
rhos = np.array(rhos)
axs[1].plot(np.median(rhos,axis=0), color='orange', label=f'Median value ({iters} iters)')
axs[1].fill_between(range(T+1), np.median(rhos,axis=0)-np.std(rhos,axis=0),
                np.median(rhos,axis=0)+np.std(rhos,axis=0),
                color='orange', alpha=.3, label=fr'$\pm 1\sigma$')

axs[1].legend(ncols=3, loc='upper right')
axs[1].set_xlabel("Time steps")
axs[1].set_ylabel("Fraction of living nodes")

model.diagram(ax=axs[0])
fig.tight_layout()

# problem with B_set [1,2], S_set []

In [ ]:
# show some density paths for a couple of initial conditions for a particular LLNA
T = 100

# random init config
for init_rho in tqdm(np.arange(0.1,1,.1)):
    s0 = init_config_with_dens(N, init_rho)
    s0 = tc.tensor(s0)[None,:]

    # evolve first model
    s = model3.forward(edges, s0, T=T)
    s = np.array(s)[0]
    s_mean = np.mean(s, axis=1)

    plt.plot(s_mean, label=fr'$\rho={round(init_rho,1)}$', alpha=.5)
plt.legend(ncol=3)
plt.ylabel("Density")
plt.xlabel("Time step")
plt.title("Density evolutions from different initial densities")

To do:
- Generalise formula for next-state average density
    - Handle higher degree
    - Handle higher resolution
- Check what happens for higher degrees in the ring network
    - Plot analytical curves
    - Scatter empirical findings
- 

In [ ]:
from src.analysis import _interval_encoding

degree = 7
resolution = 5
rhos = np.linspace(0, 1, degree+1)

int_enc = _interval_encoding(resolution, rhos[np.newaxis,:], iso=True)[0]

# Condition: nth column is True & all others are False
j = 2
mask = (int_enc[:, j])

# Get the row indices
row_indices = np.where(mask)[0]

binom_pdf = binom.pmf(np.arange(0, degree+1), degree, 0.5)
print(binom_pdf)
print(j)

total_prob_in_j = np.sum(binom_pdf[row_indices])
print(fr"Probability of being in R_{j}:", total_prob_in_j)

In [ ]:
model=LLNA(resolution)
model.diagram(degree=degree, plot_dist=True)